Entitas Utama:
1. Resource = Objek yang dapat dipesan oleh anggota lab. Memiliki dua jenis spesifik:
    - Workstation = Fasilitas unit komputer yang memiliki spesifikasi GPU tertentu.
    - Meeting Room = Ruangan diskusi yang memiliki batasan kapasitas maksimal peserta.
2. Member = Pengguna yang melakukan pemesanan resource.
3. Reservation = Catatan transaksi pemesanan yang menghubungkan Member dan Resource pada jadwal tertentu. Menyimpan reservation_id, waktu mulai (start_time), durasi, jumlah peserta, dan status (ACTIVE atau CANCELLED).
4. ISLabSystem = Mengelola dan menyimpan data anggota, resource, dan reservasi serta validasi.

Hubungan Transaksi Pemesanan:
1. Hubungan Jenis Resource (Resource -> Workstation & Meeting Room)
    - Pewarisan: Workstation dan Meeting Room adalah jenis spesifik dari Resource. Keduanya otomatis memiliki atribut dasar Resource (ID dan Nama),tetapi masing-masing menambahkan atribut khusus (GPU untuk Workstation, Kapasitas untuk Meeting Room).
2. Hubungan Transaksi Pemesanan (Member & Resource -> Reservation)
    - Reservation menghubungkan 1 Member dengan 1 Resource. Objek Reservation menyimpan siapa yang memesan, barang apa yang dipesan, kapan waktunya, dan berapa durasinya.
3. Hubungan Pengelolaan Sistem (ISLabSystem -> Semuanya)
    - Class ISLabSystem bertindak sebagai pusat sistem yang menyimpan dan mengelola seluruh daftar Member, Resource, dan Reservation di dalam satu tempat.

Aturan Utama:
1. Tidak Boleh Double Booking: Dua reservasi berstatus ACTIVE untuk resource yang sama tidak boleh memiliki rentang waktu yang bertabrakan.
2. Durasi Harus lebih dari 0
3. Khusus Meeting Room jumlah peserta reservasi tidak boleh melebihi kapasitas maksimal ruangan
4. Reservasi ACTIVE hanya bisa di batalkan satu kali saja menjadi CANCELLED

---

Object Design 

1. Class Member
    - Tanggung jawab: Menyimpan data identitas anggota laboratorium, seperti member_id, nama, dan role.
2. Class Resource
    - Tanggung jawab: Menjadi kelas induk untuk semua fasilitas lab. Menyimpan data dasar seperti resource_id dan nama fasilitas, serta menyediakan method validate_booking().
3. Class Workstation
    - Tanggung jawab: Menangani fasilitas komputer. Mewarisi sifat Resource dan menyimpan atribut khusus berupa spesifikasi GPU.
4. Class MeetingRoom 
    - Tanggung jawab: Menangani ruangan diskusi. Mewarisi sifat Resource, menyimpan batasan kapasitas ruangan (capacity), dan memvalidasi agar jumlah peserta tidak melebihi kapasitas.
5. Class Reservation
    - Tanggung jawab: Mengelola transaksi pemesanan fasilitas. Menghubungkan Member dan Resource, menghitung waktu selesai pemesanan, mengecek bentrokan jadwal, dan menangani perubahan status pembatalan.
6. Class ISLabSystem
    - Tanggung jawab: Bertindak sebagai pengelola utama seluruh sistem. Menyimpan daftar member, resource, dan reservasi di dalam dictionary, serta menyediakan fungsi untuk membuat reservasi, membatalkan reservasi, menampilkan daftar reservasi, dan mengecek ketersediaan resource.

Hubungan Antar Class
1. Workstation dan MeetingRoom adalah child class dari Resource. Keduanya menerima atribut dasar dari Resource dan menambahkan aturan/atribut spesifiknya masing-masing.
2. Reservation memiliki hubungan dengan Member dan Resource. Setiap 1 objek Reservation menyimpan referensi ke 1 objek Member (pemesan) dan 1 objek Resource (fasilitas yang dipesan).
3. ISLabSystem menjadi wadah yang menyimpan banyak objek Member, Resource, dan Reservation di dalam struktur data internalnya.


In [ ]:
from datetime import datetime, timedelta

class Member:
    def __init__(self, member_id: str, name: str, role: str):
        self.member_id = member_id
        self.name = name
        self.role = role

class Resource:
    def __init__(self, resource_id: str, name: str):
        self.resource_id = resource_id
        self.name = name
    
    def validate_booking(self, participants: int = 1):
        pass

    def __repr__(self):
        return f"{self.resource_id} | {self.name}"

class Workstation(Resource):
    def __init__(self, resource_id: str, name: str, gpu_spec):
        super().__init__(resource_id, name)
        self.gpu_spec = gpu_spec

class MeetingRoom(Resource):
    def __init__(self, resource_id: str, name: str, capacity: int):
        super().__init__(resource_id, name)
        self.capacity = capacity

    def validate_booking(self, participants: int = 1):
        if participants > self.capacity:
            raise ValueError(f"Jumlah peserta ({participants}) melebihi kapasitas ruangan ({self.capacity}).")

class Reservation:
    def __init__(self, reservation_id: str, member: Member, resource: Resource, start_time: datetime, duration: float, participants: int = 1):
        if duration <=0:
            raise ValueError("Durasi reservasi harus lebih dari 0 jam")
        
        self.reservation_id = reservation_id
        self.member = member
        self.resource = resource
        self.start_time = start_time
        self.duration = duration
        self.participants = participants
        self.status = "ACTIVE"
    
    @property
    def end_time(self) -> datetime:
        return self.start_time + timedelta(hours=self.duration)
    
    def is_overlap(self, start_time: datetime, duration: float) -> bool:
        if self.status != "ACTIVE":
            return False
        other_end = start_time + timedelta(hours=duration)

        return max(self.start_time, start_time) < min(self.end_time, other_end)
    
    def cancel(self):
        if self.status == "CANCELLED":
            raise ValueError("Reservasi ini sudah dibatalkan sebelumnya")
        self.status = "CANCELLED"

    def __repr__(self):
        start_fmt = self.start_time.strftime("%d %b %Y %H:%M")
        end_fmt = self.end_time.strftime("%H:%M")

        return f"[{self.reservation_id}] {self.status} | {self.member.name} -> {self.resource.name} ({start_fmt} - {end_fmt})"

class ISLabSystem:
    def __init__(self):
        self.members = {}
        self.resources = {}
        self.reservations = {}
    
    def add_member(self, member: Member):
        self.members[member.member_id] = member
    
    def add_resource(self, resource: Resource):
        self.resources[resource.resource_id] = resource
    
    def create_reservation(self, reservation_id: str, member_id: str, resource_id: str, start_time: datetime, duration: float, participants: int = 1):
        if member_id not in self.members:
            raise ValueError(f"Member ID '{member_id}' tidak ditemukan")
        if resource_id not in self.resources:
            raise ValueError(f"Resource ID '{resource_id}' tidak ditemukan")
        
        member = self.members[member_id]
        resource = self.resources[resource_id]

        resource.validate_booking(participants)

        for res in self.reservations.values():
            if res.resource.resource_id == resource_id and res.is_overlap(start_time, duration):
                raise ValueError(f"Jadwal bentrok dengan reservasi {res.reservation_id} ({res.start_time.strftime('%H:%M')}-{res.end_time.strftime('%H:%M')}).")
        
        res = Reservation(reservation_id, member, resource, start_time, duration, participants)
        self.reservations[reservation_id] = res
        return res
    
    def cancel_reservation(self, reservation_id: str):
        if reservation_id not in self.reservations:
            raise ValueError(f"Reservasi '{reservation_id}' tidak ditemukan")
        self.reservations[reservation_id].cancel()
    
    def get_reservations(self, member_id: str = None, resource_id: str = None):
        results = list(self.reservations.values())
        if (member_id):
            results = [r for r in results if r.member.member_id == member_id]
        if (resource_id):
            results = [r for r in results if r.resource.resource_id == resource_id]
        return results

    def get_available_resources(self, start_time: datetime, duration: float):
        available = []
        for res_id, resource in self.resources.items():
            has_overlap = any(
                res.is_overlap(start_time, duration)
                for res in self.reservations.values()
                if res.resource.resource_id == res_id
            )
            if not has_overlap:
                available.append(resource)
        return available

In [2]:
#Skenario Wajib

lab = ISLabSystem()

lab.add_resource(Workstation("WS01", "AI Workstation 1", "RTX 4090"))
lab.add_resource(MeetingRoom("MR01", "Discussion Room", capacity=8))

lab.add_member(Member("M001", "Andi", "Student"))
lab.add_member(Member("M002", "Sarah", "Assistant"))
lab.add_member(Member("M003", "Budi", "Student"))

try:
    res1 = lab.create_reservation("R001", "M001", "WS01", datetime(2026, 9, 10, 9, 0), duration=2)
    print("Berhasil:", res1)
except ValueError as e:
    print("Gagal:", e)

try:
    res2 = lab.create_reservation("R002", "M002", "WS01", datetime(2026, 9, 10, 10, 0), duration=2)
    print("Berhasil:", res2)
except ValueError as e:
    print("Gagal:", e)

try:
    res3 = lab.create_reservation("R003", "M002", "WS01", datetime(2026, 9, 10, 11, 0), duration=1)
    print("Berhasil:", res3)
except ValueError as e:
    print("Gagal:", e)

try:
    res4 = lab.create_reservation("R004", "M003", "MR01", datetime(2026, 9, 11, 13, 0), duration=2, participants=10)
    print("Berhasil:", res4)
except ValueError as e:
    print("Gagal:", e)

try:
    lab.cancel_reservation("R001")
    print("Berhasil: Status menjadi->", lab.reservations["R001"].status)
except ValueError as e:
    print("Gagal:", e)

Berhasil: [R001] ACTIVE | Andi -> AI Workstation 1 (10 Sep 2026 09:00 - 11:00)
Gagal: Jadwal bentrok dengan reservasi R001 (09:00-11:00).
Berhasil: [R003] ACTIVE | Sarah -> AI Workstation 1 (10 Sep 2026 11:00 - 12:00)
Gagal: Jumlah peserta (10) melebihi kapasitas ruangan (8).
Berhasil: Status menjadi-> CANCELLED


In [3]:
#Edge Case Tambahan
try:
    lab.create_reservation("RE01", "M001", "WS01", datetime(2026, 9, 12, 10, 0), duration=-1)
except ValueError as e:
    print("Gagal:", e)

try:
    lab.cancel_reservation("R001")
except ValueError as e:
    print("Gagal:", e)

try:
    lab.create_reservation("RE03", "M999", "WS01", datetime(2026, 9, 12, 10, 0), duration=1)
except ValueError as e:
    print("Gagal:", e)  

available = lab.get_available_resources(datetime(2026, 9, 10, 9, 0), duration=2)
print(f"Ruang yang tersedia: {[r.name for r in available]}")

Gagal: Durasi reservasi harus lebih dari 0 jam
Gagal: Reservasi ini sudah dibatalkan sebelumnya
Gagal: Member ID 'M999' tidak ditemukan
Ruang yang tersedia: ['AI Workstation 1', 'Discussion Room']


##### Hal Yang Perlu Dikembangkan

Jika sistem ini dikembangkan menjadi sistem reservasi yang digunakan IS Lab ada beberapa hal yang perlu dikembangkan yaitu lebih tepatnya menambahkan database relasional deperti mysql atau postgresql agar datanya tersimpan permanen. Dan juga bisa dikembangkan di segi pembuatan tampilan menjadi website atau aplikasi dan integrasi pakai API agar lebih mudah di gunakan
